In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
df = pd.read_csv("house_prices.csv")
display(df.head())

,area_sqft,bedrooms,age_years,distance_km,price_inr
0,600,1,15,8,3000000
1,750,2,10,7,3800000
2,900,2,8,6,4600000
3,1000,2,5,5,5200000
4,1100,3,7,5,5700000


In [4]:
print("Shape:", df.shape)

df.info()

display(df.describe())

print("Missing values:")
print(df.isnull().sum())

print("Duplicate rows:", df.duplicated().sum())

Shape: (20, 5)
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   area_sqft    20 non-null     int64
 1   bedrooms     20 non-null     int64
 2   age_years    20 non-null     int64
 3   distance_km  20 non-null     int64
 4   price_inr    20 non-null     int64
dtypes: int64(5)
memory usage: 932.0 bytes


,area_sqft,bedrooms,age_years,distance_km,price_inr
count,20.000000,20.000000,20.00000,20.000000,2.000000e+01
mean,1595.000000,3.250000,6.10000,4.150000,8.300000e+06
std,641.728751,1.118034,4.91935,2.455391,3.485911e+06
min,600.000000,1.000000,0.00000,1.000000,3.000000e+06
25%,1075.000000,2.750000,2.00000,2.000000,5.575000e+06
50%,1575.000000,3.000000,5.00000,4.000000,8.250000e+06
75%,2025.000000,4.000000,8.50000,6.000000,1.062500e+07
max,2800.000000,5.000000,18.00000,9.000000,1.480000e+07


Missing values:
area_sqft      0
bedrooms       0
age_years      0
distance_km    0
price_inr      0
dtype: int64
Duplicate rows: 0


In [5]:
X = df[["area_sqft", "bedrooms", "age_years", "distance_km"]]
y = df["price_inr"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (20, 4)
y shape: (20,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 15
Testing rows: 5


In [7]:
model = LinearRegression()

model.fit(X_train, y_train)

print("Model training completed successfully.")
print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)

Model training completed successfully.
Intercept: 1308822.4688777756
Coefficients: [   4820.25891358    7383.18760497   21856.69047039 -213667.96369184]


In [8]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Actual Price": y_test,
    "Predicted Price": y_pred
})

results["Residual"] = results["Actual Price"] - results["Predicted Price"]
results["Absolute Error"] = results["Residual"].abs()

display(results.round(2))

,Actual Price,Predicted Price,Residual,Absolute Error
0,3000000,2826867.65,173132.35,173132.35
17,6200000,6363913.16,-163913.16,163913.16
15,14800000,14415127.44,384872.56,384872.56
1,3800000,3661674.19,138325.81,138325.81
8,8800000,8687108.73,112891.27,112891.27


In [9]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE: ₹{mae:,.2f}")
print(f"MSE: {mse:,.2f}")
print(f"RMSE: ₹{rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

MAE: ₹194,627.03
MSE: 47,369,538,452.11
RMSE: ₹217,645.44
R² Score: 0.9974


In [10]:
largest_error = results["Absolute Error"].idxmax()

print("House with the largest absolute error:")
display(results.loc[[largest_error]])

print("Positive residuals:", (results["Residual"] > 0).sum())
print("Negative residuals:", (results["Residual"] < 0).sum())
print("Zero residuals:", (results["Residual"] == 0).sum())

House with the largest absolute error:


,Actual Price,Predicted Price,Residual,Absolute Error
15,14800000,1.441513e+07,384872.562469,384872.562469


Positive residuals: 4
Negative residuals: 1
Zero residuals: 0


In [11]:
baseline_prediction = np.full(len(y_test), y_train.mean())

baseline_mae = mean_absolute_error(y_test, baseline_prediction)
baseline_mse = mean_squared_error(y_test, baseline_prediction)
baseline_rmse = np.sqrt(baseline_mse)
baseline_r2 = r2_score(y_test, baseline_prediction)

comparison = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R²"],
    "Linear Regression": [mae, mse, rmse, r2],
    "Baseline": [baseline_mae, baseline_mse, baseline_rmse, baseline_r2]
})

display(comparison.round(4))

,Metric,Linear Regression,Baseline
0,MAE,1.946270e+05,3.845333e+06
1,MSE,4.736954e+10,1.979698e+13
2,RMSE,2.176454e+05,4.449379e+06
3,R²,9.974000e-01,-9.440000e-02
